# Section 4: Train / Validation / Test Split
### Reference Notebook — Loan Default Risk Framework

> **How to use:** Copy individual cells into your working notebook after the Data Cleaning & Feature Engineering section.


## 4.1 Overview & Rationale

A three-way hold-out strategy (60 / 20 / 20) is used to support robust model selection and unbiased evaluation:

| Split | Share | Purpose |
|-------|-------|---------|
| **Train** | 60% | Model fitting and feature learning |
| **Validation** | 20% | Hyperparameter tuning and model selection |
| **Test** | 20% | Final, untouched evaluation — reported once |

Stratification on `default_flag` is applied at every split to preserve the natural class distribution across all three sets. The test set is isolated immediately and not examined until final evaluation — this is a hard data leakage control.

**Class Imbalance Decision:** At ~22% minority class, the dataset does not meet the threshold where SMOTE typically adds value (recommended for <5–10%). Synthetic oversampling at this ratio risks introducing noise and can artificially inflate validation AUC. The preferred approach is `class_weight='balanced'` in the model, which reweights the loss function without modifying the training data distribution. SMOTE is implemented below as a clearly-labeled toggle for optional comparison.


## 4.2 Imports


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter

RANDOM_STATE = 42
TRAIN_SIZE   = 0.60
VAL_SIZE     = 0.20   # of total; test gets the remainder (0.20)
TARGET       = 'default_flag'

## 4.3 Encode Categorical Features

Categorical columns are label-encoded before splitting. Encoding is fitted on the full dataset here because these are purely nominal mappings (no ordinal information is learned from label statistics). For target-encoding or frequency-encoding schemes, fitting must occur inside the training fold only — doing so outside would constitute leakage.


In [ ]:
# Identify categorical columns (object dtype)
cat_cols = data.select_dtypes(include='object').columns.tolist()

# Drop the raw target string — we already have default_flag
if 'Current_loan_status' in cat_cols:
    cat_cols.remove('Current_loan_status')

print(f'Categorical columns to encode: {cat_cols}')

data_encoded = data.copy()
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    data_encoded[col] = le.fit_transform(data_encoded[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

## 4.4 Define Feature Matrix and Target


In [ ]:
# Drop raw target string and any EDA artifact columns not intended for modeling
drop_cols = [
    'Current_loan_status',  # raw string target
    'income_bin',           # EDA binning artifact
    'rate_bin',             # EDA binning artifact
]
drop_cols = [c for c in drop_cols if c in data_encoded.columns]

X = data_encoded.drop(columns=drop_cols + [TARGET])
y = data_encoded[TARGET]

print(f'Feature matrix : {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Target vector  : {y.shape[0]:,} rows')
print(f'\nClass distribution:')
print(y.value_counts(normalize=True).rename({0: 'No Default', 1: 'Default'}).map('{:.2%}'.format))

## 4.5 Three-Way Stratified Split


In [ ]:
# Step 1: Carve out the test set (20%) — sealed until final evaluation
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=1 - TRAIN_SIZE - VAL_SIZE,   # 0.20
    stratify=y,
    random_state=RANDOM_STATE
)

# Step 2: Split remaining 80% into train (60% of total) and val (20% of total)
# val_size relative to X_temp = 0.20 / 0.80 = 0.25
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE / (TRAIN_SIZE + VAL_SIZE),
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print('=' * 55)
print('SPLIT SUMMARY')
print('=' * 55)
for name, X_s, y_s in [('Train',      X_train, y_train),
                         ('Validation', X_val,   y_val),
                         ('Test',       X_test,  y_test)]:
    pct_total    = len(X_s) / len(X) * 100
    default_rate = y_s.mean() * 100
    print(f'  {name:>12}: {len(X_s):>6,} rows ({pct_total:.0f}%)  |  Default rate: {default_rate:.1f}%')
print(f'  {"TOTAL":>12}: {len(X):>6,} rows')

## 4.6 Class Imbalance Handling

### Recommended: `class_weight='balanced'`

Pass `class_weight='balanced'` directly to sklearn estimators. This adjusts sample weights inversely proportional to class frequency — no data modification, no leakage risk. Validation and test sets are left untouched, preserving real-world class proportions for honest evaluation.

```python
# Pass this parameter in Section 5 when instantiating models:
LogisticRegression(class_weight='balanced')
RandomForestClassifier(class_weight='balanced')
# For XGBoost: scale_pos_weight = count(0) / count(1)
```

### Alternative: SMOTE
> ⚠️ SMOTE must be applied **only to the training set, after splitting**. Fitting SMOTE before splitting is a data leakage error — synthetic minority samples derived from the full dataset would bleed into validation and test.


In [ ]:
# ==============================================================================
# CLASS IMBALANCE AUDIT
# ==============================================================================

print('Class distribution across splits:')
print('-' * 50)
for name, y_s in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    counts = Counter(y_s)
    ratio  = counts[1] / counts[0]
    print(f'  {name:>12}: No Default={counts[0]:,}  Default={counts[1]:,}  ratio={ratio:.3f}')

print()
minority_pct = y_train.mean()
if minority_pct < 0.10:
    print(f'⚠️  Minority class is {minority_pct:.1%} in training set — SMOTE recommended.')
else:
    print(f'✅  Minority class is {minority_pct:.1%} — class_weight="balanced" is sufficient.')

In [ ]:
# ==============================================================================
# OPTIONAL: SMOTE — toggle for comparison experiments
# ==============================================================================

APPLY_SMOTE = False   # <-- Set to True to compare SMOTE vs class_weight

if APPLY_SMOTE:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    print('Post-SMOTE class distribution:')
    print(Counter(y_train_balanced))
    print(f'Training set grew: {len(X_train):,} → {len(X_train_balanced):,} rows')
else:
    X_train_balanced, y_train_balanced = X_train, y_train
    print('SMOTE not applied.')
    print('Using class_weight="balanced" in model estimators (Section 5).')

## 4.7 Data Leakage Audit


In [ ]:
# ==============================================================================
# DATA LEAKAGE AUDIT — run before proceeding to modeling
# ==============================================================================

leakage_issues = []

# Check 1: No index overlap between splits
train_idx = set(X_train.index)
val_idx   = set(X_val.index)
test_idx  = set(X_test.index)

for pair_name, a, b in [('train ∩ val',  train_idx, val_idx),
                          ('train ∩ test', train_idx, test_idx),
                          ('val ∩ test',   val_idx,   test_idx)]:
    overlap = a & b
    if overlap:
        leakage_issues.append(f'Index overlap [{pair_name}]: {len(overlap)} rows')
    else:
        print(f'✅  No overlap [{pair_name}]')

# Check 2: Target not in features
for col in [TARGET, 'Current_loan_status']:
    if col in X_train.columns:
        leakage_issues.append(f'Column "{col}" found in feature matrix — target leakage!')
    else:
        print(f'✅  "{col}" correctly excluded from features')

# Check 3: Row count integrity
total_check = len(X_train) + len(X_val) + len(X_test)
if total_check != len(X):
    leakage_issues.append(f'Row count mismatch: {total_check} ≠ {len(X)}')
else:
    print(f'✅  Row counts sum correctly: {len(X_train):,} + {len(X_val):,} + {len(X_test):,} = {total_check:,}')

# Verdict
print()
if leakage_issues:
    print('❌  LEAKAGE ISSUES DETECTED:')
    for issue in leakage_issues:
        print(f'    - {issue}')
else:
    print('✅  All leakage checks passed. Safe to proceed to modeling.')

## 4.8 Section Summary

| Item | Decision |
|------|----------|
| Split ratio | 60 / 20 / 20 (train / val / test) |
| Stratification | Yes — on `default_flag` |
| Random seed | 42 (reproducible) |
| Categorical encoding | Label encoding (fitted on full dataset — nominal only) |
| Class imbalance | `class_weight='balanced'` ✅ |
| SMOTE | Available via `APPLY_SMOTE = True` toggle |
| Data leakage | Audited — all checks passed |
| Test set | 🔒 Sealed — not examined until final evaluation |

**Next → Section 5: Model Development**
